In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import root_mean_squared_error
from sklearn.preprocessing import StandardScaler

In [2]:
data = pd.read_csv("../Merge/final_selected_data.csv")
data.head()

,room_type_id,large_double_bed,large_bed,single_bed,sofa_bed,double_bed,small_double_bed,king_size_bed,futon_mattress,bunk_bed,...,wifi_miễn_phí,không_hoàn_tiền,miễn_phí_hủy,vào_hồ_bơi_miễn_phí,đã_kèm_bữa_sáng,hotel_id,room_room_type_name,hotel_name,hotel_address,region
0,1,1,0,0,0,0,0,0,0,0,...,1,0,1,0,0,71897952,Phòng Deluxe Có Giường Cỡ King (Deluxe King Room),Nha Nghi Nhung - Nhung Motel,"73 Đoàn Thị Điểm, Bà Rịa, Bà Rịa, Việt Nam",Bà Rịa
1,2,0,1,0,0,0,0,0,0,0,...,1,0,1,0,0,49685029,Phòng Tiêu Chuẩn (Standard Room),Nhà nghỉ Ruby Bà Rịa (Ruby Motel Bà Rịa),"KDC Baria City Gate, Long Huong Ward, Ba Ria C...",Bà Rịa
2,3,0,1,0,0,0,0,0,0,0,...,1,0,1,0,0,49685029,Phòng gia đình có ban công (Family Room with B...,Nhà nghỉ Ruby Bà Rịa (Ruby Motel Bà Rịa),"KDC Baria City Gate, Long Huong Ward, Ba Ria C...",Bà Rịa
3,4,0,1,0,0,0,0,0,0,0,...,0,0,1,0,0,65481766,Phòng Có Giường Cỡ King Với Ban Công (King Roo...,Baly Hotel Bà Rịa City (Baly Hotel Ba Ria City),"QL51, Bà Rịa, Bà Rịa, Việt Nam",Bà Rịa
4,5,1,0,1,0,0,0,0,0,0,...,1,0,0,0,0,5808626,Phòng Studio Executive (Studio Executive),Citadines Central Bình Dương (Citadines Centra...,"Số 328C, Đại lộ B nh Dương, Khu phố Hưng Lộc, ...",Bình Dương


In [3]:
data2 = data.drop(columns=[
    'hotel_id',
    'hotel_name',
    'room_room_type_name',
    'hotel_address',
    'room_type_id', 
    'views', 'region'
])
data2.head()

,large_double_bed,large_bed,single_bed,sofa_bed,double_bed,small_double_bed,king_size_bed,futon_mattress,bunk_bed,flexibility_score,...,price_option_price,adults_number,children_number,bãi_đậu_xe,phòng_tập,wifi_miễn_phí,không_hoàn_tiền,miễn_phí_hủy,vào_hồ_bơi_miễn_phí,đã_kèm_bữa_sáng
0,1,0,0,0,0,0,0,0,0,1,...,289522.0,2,0,1,0,1,0,1,0,0
1,0,1,0,0,0,0,0,0,0,1,...,361111.0,2,0,1,0,1,0,1,0,0
2,0,1,0,0,0,0,0,0,0,1,...,601852.0,4,2,1,0,1,0,1,0,0
3,0,1,0,0,0,0,0,0,0,1,...,425926.0,2,0,1,0,0,0,1,0,0
4,1,0,1,0,0,0,0,0,0,2,...,1450000.0,2,0,0,0,1,0,0,0,0


In [4]:
for col in data2.columns:
    print(f"{col}:")
    print(data2[col].unique())
    print("-" * 40)

large_double_bed:
[1 0]
----------------------------------------
large_bed:
[0 1]
----------------------------------------
single_bed:
[0 1]
----------------------------------------
sofa_bed:
[0 1]
----------------------------------------
double_bed:
[0 1]
----------------------------------------
small_double_bed:
[0 1]
----------------------------------------
king_size_bed:
[0 1]
----------------------------------------
futon_mattress:
[0 1]
----------------------------------------
bunk_bed:
[0 1]
----------------------------------------
flexibility_score:
[1 2 3 5]
----------------------------------------
sqm:
[1.800e+01 3.000e+01 4.500e+01 2.000e+01 3.500e+01 5.000e+01 5.400e+01
 6.000e+01 6.300e+01 7.100e+01 2.100e+01 3.200e+01 2.500e+01 4.000e+01
 2.200e+01 1.200e+01 2.600e+01 3.800e+01       nan 3.300e+01 4.300e+01
 7.000e+01 5.200e+01 6.200e+01 7.800e+01 3.400e+01 1.700e+01 4.100e+01
 2.800e+01 7.500e+01 6.600e+01 1.000e+01 2.400e+01 1.600e+01 5.800e+01
 1.500e+01 5.500e+01 6.50

In [5]:
cols_to_scale = [
    'sqm',
    'price_option_price',
    'adults_number',
    'bathroom_count',
    'flexibility_score',
    'bedroom_count',
    'children_number'
]

scaler = StandardScaler()

data2[cols_to_scale] = scaler.fit_transform(data2[cols_to_scale])

In [6]:
print(data2['sqm'].mean())      # ≈ 0
print(data2['sqm'].var())

3.305948255447458e-17
1.0000553924555475


In [7]:
#data2.to_csv('preprocess_final_totrain_rforest.csv')
#data2.head()

In [8]:
data2['price_option_price'] = data2['price_option_price'].fillna(data2['price_option_price'].median())
data2['sqm'] = data2['sqm'].fillna(data2['sqm'].median())

In [9]:
x_data = data2.drop(columns=['price_option_price'])
y_data = data2['price_option_price']
x_data

,large_double_bed,large_bed,single_bed,sofa_bed,double_bed,small_double_bed,king_size_bed,futon_mattress,bunk_bed,flexibility_score,...,bedroom_count,adults_number,children_number,bãi_đậu_xe,phòng_tập,wifi_miễn_phí,không_hoàn_tiền,miễn_phí_hủy,vào_hồ_bơi_miễn_phí,đã_kèm_bữa_sáng
0,1,0,0,0,0,0,0,0,0,-0.226625,...,-0.281742,-0.269930,-0.646431,1,0,1,0,1,0,0
1,0,1,0,0,0,0,0,0,0,-0.226625,...,-0.281742,-0.269930,-0.646431,1,0,1,0,1,0,0
2,0,1,0,0,0,0,0,0,0,-0.226625,...,-0.281742,0.870429,2.127743,1,0,1,0,1,0,0
3,0,1,0,0,0,0,0,0,0,-0.226625,...,-0.281742,-0.269930,-0.646431,1,0,0,0,1,0,0
4,1,0,1,0,0,0,0,0,0,4.232766,...,-0.281742,-0.269930,-0.646431,0,0,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19515,0,0,1,0,0,0,0,0,0,-0.226625,...,-0.281742,-0.269930,-0.646431,1,0,1,0,1,0,0
19516,0,0,1,0,0,0,0,0,0,-0.226625,...,-0.281742,-0.269930,-0.646431,1,0,1,0,1,0,0
19517,0,0,0,0,1,0,0,0,0,4.232766,...,-0.281742,0.870429,-0.646431,1,0,1,0,1,0,0
19518,1,0,0,0,0,0,0,0,0,-0.226625,...,-0.281742,-0.269930,-0.646431,1,0,1,0,1,0,0


In [10]:
y_data

0       -0.216136
1       -0.204025
2       -0.163296
3       -0.193059
4       -0.019804
           ...   
19515   -0.227807
19516   -0.219289
19517   -0.218832
19518   -0.230825
19519   -0.217107
Name: price_option_price, Length: 19520, dtype: float64

In [11]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(x_data, y_data, test_size=0.25, random_state=42)

In [12]:
X_train.shape

(14640, 43)

In [13]:
X_test.shape

(4880, 43)

In [14]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import RandomizedSearchCV

hist_gboost = HistGradientBoostingRegressor(loss='squared_error', random_state=42)

param_dist = {
    'learning_rate': np.linspace(0.03, 0.15, 10),
    'max_iter': [300, 600, 1000, 1600],
    'max_depth': [None, 6, 8, 10],
    'max_leaf_nodes': [15, 31, 63, 127],
    'min_samples_leaf': [20, 50, 100],
    'l2_regularization': np.logspace(-2, 1, 5)
}

hist_gboost_optim = RandomizedSearchCV(
    hist_gboost,
    param_distributions=param_dist,
    n_iter=40,
    scoring='neg_root_mean_squared_error',
    cv=5,
    n_jobs=-1,
    random_state=42,
    verbose=2
)

hist_gboost_optim.fit(X_train, y_train)

Fitting 5 folds for each of 40 candidates, totalling 200 fits


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",HistGradientB...ndom_state=42)
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'l2_regularization': array([ 0.01 ... 10. ]), 'learning_rate': array([0.03 ..., 0.15 ]), 'max_depth': [None, 6, ...], 'max_iter': [300, 600, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",40
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'neg_root_mean_squared_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-val

In [15]:
#R2 score
hist_gboost_optim.score(X_test, y_test)

-0.7732064493704627

In [16]:

y_pred = hist_gboost_optim.predict(X_test)
mean_squared_error(y_test, y_pred)
#MSE score

0.5978482133480779

In [17]:
#MAE score
mean_absolute_error(y_test, y_pred)

0.15579486433042264

In [18]:
#RMSE score
root_mean_squared_error(y_test, y_pred)

0.7732064493704627

In [19]:
from sklearn.ensemble import AdaBoostRegressor
ada_gboost = AdaBoostRegressor()
param_dist_2 = {
    'n_estimators': np.arange(50, 500, 50),
    'learning_rate': np.linspace(0.01, 0.15, 10),
    'loss': ['linear', 'square', 'exponential'],
}

ada_gboost_optim = RandomizedSearchCV(
    ada_gboost,
    param_distributions=param_dist_2,
    n_iter=40,
    scoring='neg_root_mean_squared_error',
    cv=5,
    n_jobs=-1,
    random_state=42,
    verbose=2
)
ada_gboost_optim.fit(X_train, y_train)

Fitting 5 folds for each of 40 candidates, totalling 200 fits


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",AdaBoostRegressor()
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'learning_rate': array([0.01 ..., 0.15 ]), 'loss': ['linear', 'square', ...], 'n_estimators': array([ 50, 1...50, 400, 450])}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",40
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'neg_root_mean_squared_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used h

In [20]:
y_pred_ada = ada_gboost_optim.predict(X_test)
print(mean_squared_error(y_test, y_pred_ada)) #MSE
print(mean_absolute_error(y_test, y_pred_ada)) #MAE
print(ada_gboost_optim.score(X_test, y_test)) #R2
print(root_mean_squared_error(y_test, y_pred_ada))#RMSE score

1.5539299426684592
0.21776245632158792
-1.246567263595695
1.246567263595695


In [21]:
from sklearn.tree import DecisionTreeRegressor
decision_tree = DecisionTreeRegressor(criterion = 'squared_error', splitter='random')
decision_tree.fit(X_train, y_train)

,"criterion criterion: {""squared_error"", ""friedman_mse"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in the half mean Poisson deviance to find splits... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 0.24 Poisson deviance criterion.",'squared_error'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'random'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.For an example of how ``max_depth`` influences the model, see:ref:`sphx_glr_auto_examples_tree_plot_tree_regression.py`.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary ` for details.",None
,"m

In [22]:
y_pred_decisiontree = decision_tree.predict(X_test)
print(decision_tree.score(X_test, y_test)) #R2 score 
print(mean_absolute_error(y_test,y_pred_decisiontree)) #MAE
print(mean_squared_error(y_test, y_pred_decisiontree))#MSE
print(root_mean_squared_error(y_test, y_pred_decisiontree))#RMSE score

0.1087066347876322
0.1357847468807058
0.6185905357753833
0.7865052674810152


In [23]:
from sklearn.linear_model import LinearRegression
linear_regression = LinearRegression(n_jobs = -1).fit(X_train, y_train)
y_pred_linearregression = linear_regression.predict(X_test)
print(linear_regression.score(X_test, y_test)) #R2 score 
print(mean_absolute_error(y_test,y_pred_linearregression)) #MAE
print(mean_squared_error(y_test, y_pred_linearregression))#MSE
print(root_mean_squared_error(y_test,y_pred_linearregression))#RMSE score

0.08731030448514177
0.2071616083987533
0.6334403797684337
0.7958896781391461
